In [ ]:
%load_ext watermark


In [ ]:
import gc
import os

from hstrat import _auxiliary_lib as hstrat_aux
import numpy as np
import pandas as pd
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm

hstrat_aux.seed_random(1)


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2025-10-20-mls-strong-fossils__mdcstats__wse-async-ga")
teeplot_subdir


## Get Data


In [ ]:
df = pd.read_parquet(
    "https://osf.io/download/acptf/",
    columns=[
        "id",
        "ancestor_id",
        "trait_num0",
        "focal_trait_count",
    ],
).fillna(
    {"trait_num0": 0, "focal_trait_count": 0}
).astype(
    {
        "id": np.uint32,
        "ancestor_id": np.uint32,
        "trait_num0": np.uint32,
        "focal_trait_count": np.uint32,
    }
)
gc.collect()
df.info()


In [ ]:
df


In [ ]:
with hstrat_aux.log_context_duration("alifestd_mark_leaves", display):
    df = hstrat_aux.alifestd_mark_leaves(df, mutate=True)

with hstrat_aux.log_context_duration("alifestd_mark_roots", display):
    df = hstrat_aux.alifestd_mark_roots(df, mutate=True)

with hstrat_aux.log_context_duration(
    "alifestd_mark_num_leaves_asexual", display
):
    df = hstrat_aux.alifestd_mark_num_leaves_asexual(df, mutate=True)

with hstrat_aux.log_context_duration(
    "alifestd_mark_node_depth_asexual", display
):
    df = hstrat_aux.alifestd_mark_node_depth_asexual(df, mutate=True)

gc.collect()


In [ ]:
print(f"""
    {df["num_leaves"].max()=}
    {df.loc[df["is_root"], "num_leaves"].sum()=}
    {df["is_root"].sum()=}
""")


## Prep Data


In [ ]:
summary = []
for trait in "trait_num0", "focal_trait_count":
    print(f"Processing trait: {trait}")
    absent = df["is_leaf"].values & (df[trait].values == 0)
    present = df["is_leaf"].values & (df[trait].values != 0)

    with hstrat_aux.log_context_duration(
        "alifestd_screen_trait_defined_clades_fitch_asexual", display
    ):
        df[f"mdc_{trait}"] = (
            hstrat_aux.alifestd_screen_trait_defined_clades_fitch_asexual(
                df,
                mutate=True,
                mask_trait_absent=absent,
                mask_trait_present=present,
                progress_wrap=tqdm,
            )
        )
        summary.append(
            {
                "trait": trait,
                "mdc_count": df[f"mdc_{trait}"].sum(),
                "mdc_mean": df[f"mdc_{trait}"].mean(),
            }
        )

    with hstrat_aux.log_context_duration(
        "alifestd_calc_clade_trait_frequency_asexual", display
    ):
        df[f"freq_{trait}"] = (
            hstrat_aux.alifestd_calc_clade_trait_frequency_asexual(
                df,
                mutate=True,
                mask_trait_absent=absent,
                mask_trait_present=present,
            )
        )

    for i in range(0, 101, 25):
        df[f"mdc_f{i}pct_{trait}"] = df[f"mdc_{trait}"] & (
            df[f"freq_{trait}"] >= i / 100
        )
        summary.append(
            {
                "trait": f"{trait}_f{i}pct",
                "mdc_count": df[f"mdc_f{i}pct_{trait}"].sum(),
                "mdc_mean": df[f"mdc_f{i}pct_{trait}"].mean(),
            }
        )

    gc.collect()

pd.DataFrame(summary)


## Example Plot


In [ ]:

for f in ["f75pct_", ""]:
    data1 = df.loc[df[f"mdc_{f}focal_trait_count"], "num_leaves"].to_frame()
    data1["treatment"] = "Strong MLS"
    display(data1.describe())
    data1["one"] = 1

    data2 = df.loc[df["mdc_trait_num0"], "num_leaves"].to_frame()
    display(data2.describe())
    data2["treatment"] = "Neutral"
    data2["one"] = 1

    strip_args = dict(
        x='treatment',
        y='num_leaves',
        hue="one",
        order=['Strong MLS', 'Neutral'],
        hue_order=[1, 2],
        alpha=0.3,
        dodge=True,
        jitter=0.3,
        legend=False,
        marker="o",
        s=3,
    )
    boxen_args = dict(
        x='treatment',
        y='num_leaves',
        hue="one",
        hue_order=[2, 1],
        dodge=True,
        legend=False,
    )

    with tp.teed(
        sns.stripplot,
        data=data1.sample(n=3000, random_state=1),
        **strip_args,
        palette=["#ACC791"] * 2,
        teeplot_subdir=teeplot_subdir,
        teeplot_outattrs={"filter": f},
    ) as ax:
        sns.stripplot(
            data=data2.sample(n=3000, random_state=1),
            **strip_args,
            palette=["#C5A3FF"] * 2,
        )

        sns.boxenplot(
            data=data1,
            **boxen_args,
            ax=ax,
            palette=["#D6E5BD"] * 2,
        )
        sns.boxenplot(
            data=data2,
            **boxen_args,
            ax=ax,
            palette=["#DBCDF0"] * 2,
        )

        ax.set_yscale("log")
        ax.set_xlabel("")
        ax.set_ylabel("Clade Size (leaf count)")
        sns.despine(ax=ax)
        ax.figure.set_size_inches(2, 2.5)

    gc.collect()
